# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook provides a step-by-step workflow for loading, exploring, and processing the [FAIR² dataset on second primary colorectal cancer](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the [mlcroissant](https://github.com/mlcommons/croissant) library. The dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/), which enables structured, reproducible interaction with datasets and their metadata.

### Dataset Source

The dataset source is provided as a Croissant schema at the following URL:

- `https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install `mlcroissant` if not already present
!pip install mlcroissant --quiet

## 1. Data Loading

In this section, we load the Croissant metadata and records from the dataset using `mlcroissant`. This provides both the high-level description and structured access to underlying record sets.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL to the Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access the high-level metadata
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Identifier:  {metadata.identifier}")
print(f"Description: {metadata.description}\n")
print(f"Data published: {metadata.datePublished}")

## 2. Data Overview

Explore and list available record sets and their associated field `@id`s. In Croissant, each entity (record set, field, column) is referenced via a unique `@id`.

We will list all record sets defined in the schema, then for each record set, enumerate the available fields and columns, all referencing their `@id`s.

In [ ]:
# List all record sets with their @id and names
print("Available record sets:")
record_set_ids = []
for recset in dataset.record_sets():
    print(f"- @id: {recset['@id']} | name: {recset.get('name', '(unnamed)')}")
    record_set_ids.append(recset['@id'])

print("\nExample fields for each record set:")
for recset in dataset.record_sets():
    print(f"\nRecordSet @id: {recset['@id']} | name: {recset.get('name', '(unnamed)')}")
    # List all fields (each as a dict with @id)
    fields = recset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    if fields:
        for field in fields:
            print(f"   - Field @id: {field['@id']}")
    else:
        print("   (No fields found)")

## 3. Data Extraction

We will load actual data from a chosen record set into a pandas DataFrame for further processing. The dataset may have multiple record sets; here, we extract all and display their IDs and fields. All extractions use the canonical `@id` for precise reference.

_Below, we extract all main tabular record sets._

In [ ]:
# Build DataFrames for each record set in the dataset
dataframes = {}

print("Loading record sets...")
for recset_id in record_set_ids:
    records = list(dataset.records(record_set=recset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"Loaded record set @id: {recset_id} | records: {df.shape[0]} | fields: {df.columns.tolist()}")
    else:
        print(f"No records found for record set @id: {recset_id}")

# If more than one, select the first main tabular set for further steps
if dataframes:
    primary_recset_id = list(dataframes.keys())[0]
    print(f"\nFields in record set @id {primary_recset_id}:\n{dataframes[primary_recset_id].columns.tolist()}")
    display(dataframes[primary_recset_id].head())
else:
    print("No tabular record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)

Explore the tabular data: select a numeric field (using its `@id`), apply filtering, perform normalization, and group by a categorical variable if available. All field and group references use their `@id` for rigor.

_Note: The actual numeric and group field @id values should be replaced with the correct `@id` from the previous outputs. For purposes of demonstration, use the first numeric field found, if available._

In [ ]:
# Helper: Find a numeric column to analyze (heuristic: type numeric or field name contains 'age'/'interval')
import numpy as np

df = dataframes[primary_recset_id]
numeric_field_id = None
for col in df.columns:
    # Try to pick a likely numeric field
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
    # Fallback: try to parse
    if df[col].dropna().apply(lambda x: str(x).replace('.', '', 1).isdigit()).all():
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            numeric_field_id = col
            break
        except Exception:
            continue
if not numeric_field_id:
    raise ValueError("No numeric field found in DataFrame for demonstration.")
print(f"Using numeric field (column @id): {numeric_field_id}\n")

# Choose a group/categorical field (not equal to the numeric field)
group_field_id = None
for col in df.columns:
    if col != numeric_field_id and df[col].nunique() < 10:
        group_field_id = col
        break
if not group_field_id:
    group_field_id = df.columns[0] if df.columns[0] != numeric_field_id else (df.columns[1] if len(df.columns)>1 else None)

# Filter: Show records with numeric_field above a threshold (here, use its 25th percentile as example)
threshold = df[numeric_field_id].quantile(0.25)
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize field (Z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouped mean by the group field (if present)
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id}, with mean {numeric_field_id}:")
    display(grouped_df.head())

## 5. Visualization

Plot the distribution of the selected numeric field and compare means by group (if applicable).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# Plot grouped means if applicable
if group_field_id:
    plt.figure(figsize=(8,4))
    sns.barplot(
        data=filtered_df,
        x=group_field_id,
        y=numeric_field_id
    )
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated loading and exploring a FAIR dataset defined by the Croissant schema using the `mlcroissant` library. We:
- Loaded dataset metadata and described its provenance
- Enumerated available record sets and their fields by `@id`
- Extracted tabular data and demonstrated selection of numeric and categorical fields by `@id`
- Performed standard EDA and basic visualization on these fields

This workflow provides a starting point for in-depth analysis or modeling using standardized biomedical datasets with robust metadata.